# 01 · The RL Problem
**FOUNDATIONS** · ⏱ 25 min · 🧪 2 exercises · ⭐ 200 XP

*Agents, environments, rewards, returns and discounting — the vocabulary everything else is built on.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "01-the-rl-problem"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Supervised learning is told the right answer for every example. Reinforcement learning is
only told *how good* the outcome was — often much later, and never why. That single difference
creates every hard problem in this course: **delayed credit**, **exploration**, and
**non-stationary data** produced by the learner's own behaviour.

## 01 · The agent–environment loop

At each time step *t*:

1. the agent observes a **state** *Sₜ*
2. it picks an **action** *Aₜ* according to its **policy** *π(a | s)*
3. the environment returns a **reward** *Rₜ₊₁* and the next state *Sₜ₊₁*

```text
S₀, A₀, R₁, S₁, A₁, R₂, S₂, A₂, R₃, …
```

The interaction is often split into **episodes** (a game, a maze run) that end in a
*terminal* state. Some tasks are **continuing** and never end.

> **ℹ️ NOTE** · **Gymnasium-style API**
>
> Every environment in this course uses the same interface as the popular Gymnasium library:
> `obs, info = env.reset()` then `obs, reward, terminated, truncated, info = env.step(action)`.
> `terminated` means the episode truly ended (goal, crash); `truncated` means we cut it off
> (time limit). The difference matters later when we bootstrap.

## 02 · Rewards and the reward hypothesis

> **🔑 KEY IDEA** · **The reward hypothesis**
>
> All of what we mean by goals can be thought of as maximising the expected cumulative sum of a
> scalar reward signal.

The reward says **what** you want, not **how** to get it. Reward a chess agent for capturing
pieces and it will learn to capture pieces — even if that loses the game.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("01-the-rl-problem#0", {"q": "You reward a robot vacuum +1 for every piece of dust it sucks up. What might go wrong?", "options": ["Nothing — that is exactly the goal", "It may learn to dump dust out and re-collect it", "It will refuse to move"], "answer": 1, "explain": "Reward hacking: the agent maximises what you measured, not what you meant. Reward 'clean floor', not 'dust collected'."})

## 03 · Returns and discounting

The agent maximises the **return** — the sum of future rewards. With a **discount factor**
γ ∈ [0, 1], rewards further in the future count for less:

```text
Gₜ = Rₜ₊₁ + γ Rₜ₊₂ + γ² Rₜ₊₃ + … = Σₖ γᵏ Rₜ₊ₖ₊₁
```

The return obeys a beautiful recursion that the rest of the course leans on heavily:

```text
Gₜ = Rₜ₊₁ + γ Gₜ₊₁
```

γ near 0 makes a **myopic** agent; γ near 1 a **far-sighted** one. For continuing tasks, γ < 1
keeps the infinite sum finite. Drag the slider and watch how much a distant reward is worth.

In [ ]:
#@title 🎮 **Interactive** · Discounting explorer
rlcolab.widget("discount", gamma=0.9, rewards=[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 5])

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("01-the-rl-problem#1", {"q": "With γ = 0.5, rewards [1, 1, 1] (R₁, R₂, R₃) give a return G₀ of…", "options": ["3", "1.75", "1.5", "2.5"], "answer": 1, "explain": "1 + 0.5·1 + 0.25·1 = 1.75"})

---
### 🧪 Exercise 1 · Discounted return · ⭐ **100 XP**

Implement two functions:

- `discounted_return(rewards, gamma)` → the single number G₀ = Σ γᵏ rₖ₊₁ for a list of rewards `[R₁, R₂, …]`.
- `returns_to_go(rewards, gamma)` → a list `[G₀, G₁, …, G_{T-1}]`, the return from *every* time step.

For `returns_to_go`, use the recursion **Gₜ = Rₜ₊₁ + γ Gₜ₊₁**, sweeping *backwards*. It is O(T) instead of O(T²) — and you will reuse this exact function for Monte Carlo and REINFORCE later.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 01-the-rl-problem/01-discounted-return
import rlhud


def discounted_return(rewards, gamma):
    """G0 = r1 + gamma*r2 + gamma^2*r3 + ...   (rewards = [r1, r2, r3, ...])"""
    # TODO
    raise NotImplementedError


def returns_to_go(rewards, gamma):
    """[G0, G1, ..., G_{T-1}] computed backwards with G_t = r_{t+1} + gamma * G_{t+1}."""
    # TODO
    raise NotImplementedError


if __name__ == "__main__":
    rewards = [0, 0, 0, 1, 0, 0, 0, 0, 0, 10]
    for gamma in (0.0, 0.5, 0.9, 0.99):
        print(f"gamma={gamma:<5} G0={discounted_return(rewards, gamma):8.4f}")
        rlhud.plot(returns_to_go(rewards, gamma), label=f"γ={gamma}", title="Return-to-go per step")

In [ ]:
#@title ✅ **Check** · Discounted return (+100 XP) — tests the code you last ran above
rlcolab.check("01-the-rl-problem/01-discounted-return")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

discounted_return: loop with k = 0, 1, 2… and add gamma**k * rewards[k].


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

returns_to_go: start with G = 0 after the final step and walk the list in reverse.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

G = 0; out = []; for r in reversed(rewards): G = r + gamma * G; out.append(G) — then reverse out.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("01-the-rl-problem/01-discounted-return")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 01-the-rl-problem/01-discounted-return
# Reference solution
import rlhud


def discounted_return(rewards, gamma):
    """G0 = r1 + gamma*r2 + gamma^2*r3 + ...   (rewards = [r1, r2, r3, ...])"""
    return sum((gamma ** k) * r for k, r in enumerate(rewards))


def returns_to_go(rewards, gamma):
    """[G0, G1, ..., G_{T-1}] computed backwards with G_t = r_{t+1} + gamma * G_{t+1}."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    return out[::-1]


if __name__ == "__main__":
    rewards = [0, 0, 0, 1, 0, 0, 0, 0, 0, 10]
    for gamma in (0.0, 0.5, 0.9, 0.99):
        print(f"gamma={gamma:<5} G0={discounted_return(rewards, gamma):8.4f}")
        rlhud.plot(returns_to_go(rewards, gamma), label=f"γ={gamma}", title="Return-to-go per step")

## 04 · Policies and value functions

A **policy** π maps states to (probabilities of) actions. A **value function** says how good
it is to be somewhere *if you follow π afterwards*:

```text
v_π(s) = E_π[ Gₜ | Sₜ = s ]          q_π(s, a) = E_π[ Gₜ | Sₜ = s, Aₜ = a ]
```

Nearly every method you will build either **estimates these values** (value-based: DP, TD,
Q-learning, DQN), **optimises π directly** (policy gradient), or does both (actor-critic, PPO).

## 05 · Exploration vs exploitation

To get reward the agent should pick what has worked before (**exploit**). To find out what
works it has to try things it is unsure about (**explore**). Doing only one of these fails.
The next lesson attacks this trade-off in its purest form.

## 06 · Your first agent

Time to drive the actual capstone environment with the dumbest possible policy — random
actions — and measure how badly it does. That number is your **baseline**: every algorithm
from here on has to beat it.

---
### 🧪 Exercise 2 · The agent loop · ⭐ **100 XP**

Write `run_episode(env, policy, seed=None)` — the loop at the heart of every RL program.

1. `obs, info = env.reset(seed=seed)`
2. repeat: `action = policy(obs)`, then `obs, reward, terminated, truncated, info = env.step(action)`
3. stop when `terminated or truncated`

Return a tuple `(total_reward, n_steps)`.

Then implement `random_policy(obs)` returning a uniformly random action in `0..4` (Core Runner has 5 actions: UP, RIGHT, DOWN, LEFT, WAIT).

The demo measures the random baseline on Core Runner level 1 and records a replay.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 01-the-rl-problem/02-agent-loop
import random
import rlhud
from corerunner import CoreRunner


def run_episode(env, policy, seed=None):
    """Play one episode. Returns (total_reward, n_steps)."""
    # TODO: reset, then loop step() until terminated or truncated
    raise NotImplementedError


def random_policy(obs):
    """Ignore the observation, pick one of the 5 actions uniformly at random."""
    # TODO
    raise NotImplementedError


if __name__ == "__main__":
    random.seed(0)
    env = CoreRunner(level=1)
    totals = [run_episode(env, random_policy)[0] for _ in range(500)]
    rlhud.plot(totals, label="random agent", title="Random baseline · Core Runner L1", smooth=25)
    rlhud.stat("Mean return", sum(totals) / len(totals))
    rlhud.stat("Best return", max(totals))

    # Record one episode so you can watch it in the Visuals tab.
    obs, _ = env.reset(seed=1)
    frames = [env.snapshot()]
    done = False
    while not done:
        obs, r, term, trunc, _ = env.step(random_policy(obs))
        frames.append(env.snapshot())
        done = term or trunc
    rlhud.replay(frames, title="Random agent replay")

In [ ]:
#@title ✅ **Check** · The agent loop (+100 XP) — tests the code you last ran above
rlcolab.check("01-the-rl-problem/02-agent-loop")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Use a `while True:` loop and `break` once terminated or truncated is True.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Accumulate `total += reward` and `steps += 1` inside the loop.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

random_policy can be `return random.randrange(5)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("01-the-rl-problem/02-agent-loop")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 01-the-rl-problem/02-agent-loop
# Reference solution
import random
import rlhud
from corerunner import CoreRunner


def run_episode(env, policy, seed=None):
    """Play one episode. Returns (total_reward, n_steps)."""
    obs, info = env.reset(seed=seed)
    total, steps = 0.0, 0
    while True:
        obs, reward, terminated, truncated, info = env.step(policy(obs))
        total += reward
        steps += 1
        if terminated or truncated:
            return total, steps


def random_policy(obs):
    """Ignore the observation, pick one of the 5 actions uniformly at random."""
    return random.randrange(5)


if __name__ == "__main__":
    random.seed(0)
    env = CoreRunner(level=1)
    totals = [run_episode(env, random_policy)[0] for _ in range(500)]
    rlhud.plot(totals, label="random agent", title="Random baseline · Core Runner L1", smooth=25)
    rlhud.stat("Mean return", sum(totals) / len(totals))
    rlhud.stat("Best return", max(totals))

    # Record one episode so you can watch it in the Visuals tab.
    obs, _ = env.reset(seed=1)
    frames = [env.snapshot()]
    done = False
    while not done:
        obs, r, term, trunc, _ = env.step(random_policy(obs))
        frames.append(env.snapshot())
        done = term or trunc
    rlhud.replay(frames, title="Random agent replay")

> **💡 TIP** · **Look at the replay**
>
> After **Run**, open the *Visuals* tab: the demo records an episode of your random agent in
> Core Runner. Notice how often it wanders into a wall or a vent. We will fix that.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("01-the-rl-problem")